In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier


In [13]:
# Configure global plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

def load_and_clean_data(filepath):
    """Loads dataset and ensures chronological ordering."""
    print(f"Loading dataset from: {filepath}")
    df = pd.read_csv(filepath)
    
    # Sort by date to maintain the F1 temporal sequence
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
        
    print(f"Dataset loaded successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.")
    return df

def plot_missing_values(df):
    """0. Visualizes missing data across the dataset."""
    print("Generating Missing Values Heatmap...")
    plt.figure(figsize=(12, 6))
    
    # Calculate percentage of missing values
    missing_pct = df.isnull().mean() * 100
    missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)
    
    if len(missing_cols) > 0:
        sns.barplot(x=missing_cols.values, y=missing_cols.index, palette='Reds_r')
        plt.title('Percentage of Missing Values per Feature', fontsize=16, fontweight='bold')
        plt.xlabel('% Missing')
        plt.tight_layout()
        plt.savefig('eda_00_missing_values.png')
    else:
        print("-> No missing values detected! Skipping missing values plot.")
    plt.close()

def plot_target_distribution(df):
    """1. Analyzes Class Imbalance in the target variable."""
    print("Generating Target Distribution...")
    plt.figure(figsize=(8, 6))
    
    target_order = ['podium', 'points', 'no_points']
    colors = ['#FFD700', '#C0C0C0', '#CD7F32'] # Gold, Silver, Bronze thematic
    
    ax = sns.countplot(data=df, x='target', order=target_order, palette=colors, edgecolor='black')
    plt.title('Target Variable Distribution (Class Imbalance)', fontsize=16, fontweight='bold')
    plt.xlabel('Race Outcome')
    plt.ylabel('Count')
    
    # Annotate bar counts
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontweight='bold', xytext=(0, 5), textcoords='offset points')
        
    plt.tight_layout()
    plt.savefig('eda_01_target_distribution.png')
    plt.close()

def plot_feature_distributions(df):
    """2. Detailed histograms of continuous and window features."""
    print("Generating Continuous Feature Distributions...")
    features = [
        'driver_age', 'days_since_last_race', 'points_gap_to_leader',
        'driver_avg_position_last3', 'teammate_h2h_avg_position_delta', 'driver_dnf_rate_historical'
    ]

    axis_cfg = {
        'driver_age': {
            'xlim': (18, 45),
            'xticks': np.arange(18, 46, 2),
            'xlabel': 'Driver age (years)'
        },
        'days_since_last_race': {
            'xlim': (-2, 110),
            'xticks': [-1, 0, 7, 14, 21, 28, 56, 84, 105],
            'xlabel': 'Days since previous race (-1 = no history)'
        },
        'points_gap_to_leader': {
            'xlim': (-5, 575),
            'xticks': np.arange(0, 601, 50),
            'xlabel': 'Points behind championship leader'
        },
        'driver_avg_position_last3': {
            'xlim': (-1, 20),
            'xticks': [-1] + list(np.arange(1, 21, 1)),
            'xlabel': 'Average finishing position, previous 3 races'
        },
        'teammate_h2h_avg_position_delta': {
            'xlim': (-14, 14),
            'xticks': np.arange(-14, 15, 2),
            'xlabel': 'Average position delta vs teammate'
        },
        'driver_dnf_rate_historical': {
            'xlim': (-1, 1),
            'xticks': [-1] + list(np.round(np.arange(0, 1.01, 0.1), 1)),
            'xlabel': 'Historical DNF rate (-1 = no history)'
        }
    }

    valid_features = [f for f in features if f in df.columns]

    if valid_features:
        fig, axes = plt.subplots(2, 3, figsize=(24, 14))
        fig.suptitle('Distributions of Key Predictors (Detailed Scale)', fontsize=20, fontweight='bold', y=1.02)
        axes = axes.flatten()

        for i, col in enumerate(valid_features):
            ax = axes[i]
            values = df[col].dropna()
            cfg = axis_cfg.get(col, {})
            xlim = cfg.get('xlim')

            plot_values = values
            clipped_count = 0
            if xlim is not None:
                clipped_count = int(((values < xlim[0]) | (values > xlim[1])).sum())
                plot_values = values[(values >= xlim[0]) & (values <= xlim[1])]

            sns.histplot(plot_values, kde=True, ax=ax, color='teal', bins=60, edgecolor='black', linewidth=0.4)
            median_value = values.median()
            ax.axvline(median_value, color='crimson', linestyle='--', linewidth=1.5, label=f'Median: {median_value:.2f}')
            ax.set_title(col, fontsize=13, fontweight='bold')
            ax.set_xlabel(cfg.get('xlabel', col), fontsize=10)
            ax.set_ylabel('Frequency')
            ax.grid(True, which='major', axis='both', alpha=0.35)
            ax.grid(True, which='minor', axis='x', alpha=0.15)
            ax.minorticks_on()

            if xlim is not None:
                ax.set_xlim(*xlim)
            if 'xticks' in cfg:
                ax.set_xticks(cfg['xticks'])
                ax.tick_params(axis='x', labelrotation=45)

            stats_text = f"min={values.min():.2f}\nmax={values.max():.2f}"
            if clipped_count > 0:
                stats_text += f"\n{clipped_count} value(s) outside shown range"
            ax.text(0.98, 0.95, stats_text, transform=ax.transAxes, ha='right', va='top',
                    fontsize=9, bbox=dict(facecolor='white', edgecolor='gray', alpha=0.85))
            ax.legend(loc='upper left', fontsize=9, frameon=True)

        for j in range(i + 1, len(axes)):
            fig.delaxes(axes[j])

        plt.tight_layout()
        plt.savefig('eda_02_feature_distributions.png', bbox_inches='tight')
        plt.close()

def plot_feature_separability(df):
    """3. Boxplots to see how features separate the Target classes."""
    print("Generating Feature Separability (Boxplots)...")
    
    features_to_check = ['driver_avg_position_last3','driver_avg_position_last5','driver_avg_position_last10', 'teammate_h2h_avg_position_delta']
    target_order = ['podium', 'points', 'no_points']
    
    for feature in features_to_check:
        if feature in df.columns:
            plt.figure(figsize=(10, 6))
            
            # Swapped violinplot for boxplot
            sns.boxplot(data=df, x='target', y=feature, order=target_order, palette='Set2')
            
            plt.title(f'Target Separation by: {feature} (Boxplot)', fontsize=16, fontweight='bold')
            plt.xlabel('Race Outcome')
            plt.ylabel(feature)
            
            plt.tight_layout()
            plt.savefig(f'eda_03_separability_{feature}.png')
            plt.close()

def plot_correlation_heatmap(df):
    """4. Spearman Rank Correlation Heatmap for feature collinearity."""
    print("Generating Spearman Correlation Heatmap...")
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Drop IDs to prevent meaningless correlation calculations
    ignore_cols = ['raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'round']
    numeric_df = numeric_df.drop(columns=[c for c in ignore_cols if c in numeric_df.columns], errors='ignore')
    
    corr_matrix = numeric_df.corr(method='spearman')
    
    plt.figure(figsize=(20, 16))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    
    sns.heatmap(corr_matrix, mask=mask, cmap='RdBu_r', center=0, vmax=1, vmin=-1, 
                annot=False, square=True, linewidths=.5, cbar_kws={"shrink": .7})
    
    plt.title('Spearman Rank Correlation Heatmap (Feature Collinearity)', fontsize=20, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_04_correlation_heatmap.png', bbox_inches='tight')
    plt.close()

def plot_rf_feature_importance(df):
    """5. Baseline Random Forest to map Gini Feature Importances."""
    print("Generating Baseline Feature Importance Ranking...")
    
    # Drop target, IDs, and dates
    drop_cols = ['date', 'year', 'round', 'target', 'raceId', 'driverId', 'constructorId', 'circuitId']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    y = df['target']
    
    # Quick median imputation for missing values so the RF algorithm can run
    X = X.fillna(X.median(numeric_only=True))
    
    # Train a quick model
    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
    rf.fit(X, y)
    
    importances = rf.feature_importances_
    sorted_idx = np.argsort(importances)[::-1]
    
    top_n = min(25, len(X.columns)) # Top 25 features or less
    
    plt.figure(figsize=(12, 10))
    sns.barplot(x=importances[sorted_idx[:top_n]], y=X.columns[sorted_idx[:top_n]], palette='magma')
    
    plt.title(f'Top {top_n} Predictors (Random Forest Gini Importance)', fontsize=16, fontweight='bold')
    plt.xlabel('Gini Importance (Reduction in Impurity)')
    plt.ylabel('Features')
    
    plt.tight_layout()
    plt.savefig('eda_05_feature_importance.png')
    plt.close()

In [14]:
csv_path = '../dataset/outputs/prediction.csv'
f1_df = load_and_clean_data(csv_path)

Loading dataset from: ../dataset/outputs/prediction.csv
Dataset loaded successfully. Shape: 2454 rows, 38 columns.


In [21]:
plot_missing_values(f1_df)
plot_target_distribution(f1_df)
plot_feature_distributions(f1_df)
plot_feature_separability(f1_df)
plot_correlation_heatmap(f1_df)
plot_rf_feature_importance(f1_df)

Generating Missing Values Heatmap...
-> No missing values detected! Skipping missing values plot.
Generating Target Distribution...
Generating Continuous Feature Distributions...


/var/folders/fv/v9c75w650kqfy7c351bylhyr0000gn/T/ipykernel_19284/2588038054.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.countplot(data=df, x='target', order=target_order, palette=colors, edgecolor='black')


Generating Feature Separability (Boxplots)...


/var/folders/fv/v9c75w650kqfy7c351bylhyr0000gn/T/ipykernel_19284/2588038054.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='target', y=feature, order=target_order, palette='Set2')
/var/folders/fv/v9c75w650kqfy7c351bylhyr0000gn/T/ipykernel_19284/2588038054.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='target', y=feature, order=target_order, palette='Set2')
/var/folders/fv/v9c75w650kqfy7c351bylhyr0000gn/T/ipykernel_19284/2588038054.py:163: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='target', 

Generating Spearman Correlation Heatmap...
Generating Baseline Feature Importance Ranking...


/var/folders/fv/v9c75w650kqfy7c351bylhyr0000gn/T/ipykernel_19284/2588038054.py:217: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances[sorted_idx[:top_n]], y=X.columns[sorted_idx[:top_n]], palette='magma')
